# v4 학습 — 6클래스 (후미등 추가) / Kaggle T4

기존 5종 + **tail_light(후미등)** = 6클래스로 새로 학습.

**전역 클래스 표준 (자연어 언더바):**
| id | name |
|----|------|
| 0 | car_emblem |
| 1 | door_handle |
| 2 | fuel_cap |
| 3 | license_plate |
| 4 | side_mirror |
| 5 | tail_light |

**핵심 포인트**
- 클래스가 5→6으로 바뀌어 v3 이어받기(warm-start) 불가 → yolov8n에서 새로 학습 (백본은 사전학습 재활용)
- 후미등은 기존 6종 데이터엔 없음 → **레이 데이터(ray_all)로만** 배움 → 레이 oversampling으로 비중↑
- 기존 6종 bin은 각자 대상만 있으니 새 전역 id로 매핑
- 레이(ray_all)는 이미 전역순서(0~5) → asis

**실행 전**
- Settings → GPU T4, Internet On
- Add Input: 기존 데이터셋(bin 6개) + ray_all(.bin)  (한 데이터셋에 다 넣어도 됨)

In [ ]:
!pip install ultralytics -q

In [ ]:
# ============================================================
# 0. 입력 경로 자동 탐색
# ============================================================
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
print("input 목록:")
for p in INPUT_ROOT.iterdir():
    print(" -", p.name, "->", [f.name for f in p.iterdir()][:10])

def find_file(fname):
    hits = list(INPUT_ROOT.rglob(fname))
    return hits[0] if hits else None

BASE_DIR = None
h = find_file("handle.bin")
assert h is not None, "handle.bin 못 찾음"
BASE_DIR = h.parent
print("\n기존 데이터 폴더:", BASE_DIR)

RAY_BIN = find_file("ray_all.bin") or find_file("ray_all_remapped.bin")
assert RAY_BIN is not None, "ray_all.bin 못 찾음"
print("레이 데이터:", RAY_BIN)

POSTER_BIN = find_file("ray_poster.bin")
assert POSTER_BIN is not None, "ray_poster.bin 못 찾음"
print("포스터 데이터:", POSTER_BIN)

In [ ]:
# ============================================================
# 1. 설정
# ============================================================
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

WORK   = Path("/kaggle/working/_ft_src")
MERGED = Path("/kaggle/working/_ft_merged")

REHEARSAL_PER_CLASS = 300   # 기존 5종: 클래스당 최대
RAY_REPEAT = 8              # 레이 복제 횟수 (후미등 비중 확보). 4종 무너지면 낮추기
POSTER_REPEAT = 12          # 포스터 복제 횟수 (실전 테스트 환경이라 강하게. 실차/공공 무너지면 낮추기)
VAL_RATIO = 0.15
SEED = 42
EPOCHS = 40                 # 새 학습이라 v3(20)보다 넉넉히. patience로 조기종료
IMGSZ = 640
BATCH = 32

# 전역 표준 (자연어 언더바)
GLOBAL_NAMES = ["car_emblem", "door_handle", "fuel_cap", "license_plate", "side_mirror", "tail_light"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# 기존 bin: 각 데이터셋의 대상 클래스를 새 전역 id로 매핑
#   keep: "all" | "name:X"
#   gid : 이 소스가 전역에서 갖는 id
#   full: True=전량, False=리허설
#   repeat: 복제 횟수
SOURCES = [
    {"key":"handle",    "file":"handle.bin",    "keep":"all",           "gid":1, "full":False, "repeat":1, "dir":BASE_DIR},  # ->door_handle
    {"key":"mirror",    "file":"mirror.bin",    "keep":"name:perfect",  "gid":4, "full":False, "repeat":1, "dir":BASE_DIR},  # ->side_mirror
    {"key":"plate_old", "file":"plate_old.bin", "keep":"all",           "gid":3, "full":True,  "repeat":1, "dir":BASE_DIR},  # ->license_plate
    {"key":"plate_new", "file":"plate_new.bin", "keep":"all",           "gid":3, "full":True,  "repeat":1, "dir":BASE_DIR},
    {"key":"logo",      "file":"logo.bin",      "keep":"all",           "gid":0, "full":False, "repeat":1, "dir":BASE_DIR},  # ->car_emblem
    {"key":"fuel_cap",  "file":"fuel_cap.bin",  "keep":"name:Fuel-Cap", "gid":2, "full":False, "repeat":1, "dir":BASE_DIR},  # ->fuel_cap
    # 레이: 이미 전역순서(0~5) → asis, 전량, 복제
    {"key":"ray",       "file":RAY_BIN.name,    "keep":"asis",          "gid":-1,"full":True,  "repeat":RAY_REPEAT, "dir":RAY_BIN.parent},
    # 포스터(실전 테스트 환경): 전역순서 asis, 전량, 독립 오버샘플링
    {"key":"poster",    "file":POSTER_BIN.name, "keep":"asis",          "gid":-1,"full":True,  "repeat":POSTER_REPEAT, "dir":POSTER_BIN.parent},
]

In [ ]:
# ============================================================
# 2. .bin 압축 해제
# ============================================================
import shutil, zipfile
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
for s in SOURCES:
    src = s["dir"] / s["file"]
    assert src.exists(), f"파일 없음: {src}"
    with zipfile.ZipFile(src) as z:
        z.extractall(WORK / s["key"])
    print(f"  {s['key']:10s} <- {src.name}")

In [ ]:
# ============================================================
# 3. 라벨 정리 + 전역 id 매핑
#    - 세그멘테이션(폴리곤) 라벨이면 박스로 자동 변환
# ============================================================
import yaml

def load_names(root):
    hits = list(root.rglob("data.yaml"))
    return yaml.safe_load(open(hits[0]))["names"] if hits else []

def poly_to_box(coords):
    xs, ys = coords[0::2], coords[1::2]
    x1,x2=min(xs),max(xs); y1,y2=min(ys),max(ys)
    return (x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1

def collect_pairs(root):
    pairs=[]
    for lbl in root.rglob("*/labels/*.txt"):
        img_dir = lbl.parent.parent/"images"
        for ext in IMG_EXT:
            img=img_dir/(lbl.stem+ext)
            if img.exists(): pairs.append((img,lbl)); break
    return pairs

def process(s):
    root = WORK/s["key"]
    keep = s["keep"]
    if keep=="asis":
        # 레이: 클래스 그대로(0~5), 폴리곤이면 박스변환만
        kept=[]
        for img,lbl in collect_pairs(root):
            out=[]
            for line in open(lbl):
                p=line.split()
                if len(p)<5: continue
                c=int(p[0]); coords=list(map(float,p[1:]))
                if len(p)==5: xc,yc,w,h=coords
                else: xc,yc,w,h=poly_to_box(coords)
                out.append(f"{c} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            if out: lbl.write_text("\n".join(out)+"\n"); kept.append((img,lbl))
        print(f"  [{s['key']}] asis -> {len(kept)}장")
        return kept
    names = load_names(root)
    gid = s["gid"]
    if keep=="all":
        keep_ids=set(range(len(names)))
    else:
        cname=keep.split("name:",1)[1]
        keep_ids={names.index(cname)} if cname in names else set(range(len(names)))
        if cname not in names: print(f"  ⚠️ [{s['key']}] '{cname}' 없음->전체유지 {names}")
    print(f"  [{s['key']}] names={names} | 유지{sorted(keep_ids)} -> 전역 {gid}({GLOBAL_NAMES[gid]})")
    kept=[]
    for img,lbl in collect_pairs(root):
        out=[]
        for line in open(lbl):
            p=line.split()
            if len(p)<5: continue
            if int(p[0]) in keep_ids:
                coords=list(map(float,p[1:]))
                if len(p)==5: xc,yc,w,h=coords
                else: xc,yc,w,h=poly_to_box(coords)
                out.append(f"{gid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        if out: lbl.write_text("\n".join(out)+"\n"); kept.append((img,lbl))
    return kept

per_source={s["key"]:process(s) for s in SOURCES}
print("\n유지 이미지 수:", {k:len(v) for k,v in per_source.items()})

In [ ]:
# ============================================================
# 4. 병합 (레이 oversampling) + 클래스 분포 확인
# ============================================================
import random
if MERGED.exists(): shutil.rmtree(MERGED)
for sp in ["train","valid"]:
    (MERGED/sp/"images").mkdir(parents=True,exist_ok=True)
    (MERGED/sp/"labels").mkdir(parents=True,exist_ok=True)

def copy_pairs(pairs,sp,key,tag=""):
    for img,lbl in pairs:
        stem=f"{key}{tag}__{img.stem}"
        shutil.copy(img,MERGED/sp/"images"/(stem+img.suffix))
        shutil.copy(lbl,MERGED/sp/"labels"/(stem+".txt"))

rng=random.Random(SEED); summary={}
for s in SOURCES:
    pairs=per_source[s["key"]][:]; rng.shuffle(pairs)
    if not s["full"]: pairs=pairs[:REHEARSAL_PER_CLASS]
    n_val=max(1,int(len(pairs)*VAL_RATIO)) if pairs else 0
    copy_pairs(pairs[:n_val],"valid",s["key"])
    for r in range(s["repeat"]):
        copy_pairs(pairs[n_val:],"train",s["key"],tag=f"_r{r}")
    summary[s["key"]]=(len(pairs)-n_val)*s["repeat"]

tot_tr=len(list((MERGED/'train'/'images').iterdir()))
tot_va=len(list((MERGED/'valid'/'images').iterdir()))
print("소스별 train(복제후):", summary)
print(f"합계 train={tot_tr} valid={tot_va}  | 레이비중={summary['ray']/tot_tr*100:.1f}% | 포스터비중={summary['poster']/tot_tr*100:.1f}%")

# 클래스별 박스 분포
from collections import Counter
c=Counter()
for lbl in (MERGED/'train'/'labels').glob('*.txt'):
    for line in open(lbl):
        p=line.split()
        if p: c[int(p[0])]+=1
print("\ntrain 박스/클래스:")
for i in range(6): print(f"  {i} {GLOBAL_NAMES[i]}: {c[i]}")

cfg={"train":str(MERGED/"train"/"images"),"val":str(MERGED/"valid"/"images"),
     "nc":6,"names":GLOBAL_NAMES}
yaml.safe_dump(cfg,open(MERGED/"data.yaml","w"),allow_unicode=True)

In [ ]:
# ============================================================
# 5. 학습 (yolov8n에서 새로 - 6클래스라 v3 이어받기 불가)
#    먼저 EPOCHS=1로 시간 확인 권장
# ============================================================
from ultralytics import YOLO
model = YOLO("yolov8n.pt")   # 사전학습 백본에서 시작
model.train(
    data=str(MERGED/"data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, patience=15, seed=SEED,
    project="/kaggle/working/ft_runs", name="carparts_v4", exist_ok=True,
)

In [ ]:
# ============================================================
# 6. 클래스별 평가
# ============================================================
!yolo detect val model=/kaggle/working/ft_runs/carparts_v4/weights/best.pt data=/kaggle/working/_ft_merged/data.yaml device=0

In [ ]:
# ============================================================
# 7. 결과 (Output 패널에서 다운로드 -> models/best_v4_6cls.pt)
# ============================================================
w=Path("/kaggle/working/ft_runs/carparts_v4/weights/best.pt")
print("완료:", w, "| 존재:", w.exists(), f"| {w.stat().st_size/1e6:.1f}MB" if w.exists() else "")

---
### 학습 후
- best.pt -> 로컬 `models/best_v4_6cls.pt`
- **주의: 6클래스 모델**이라 test_video.py 결과에 tail_light가 추가로 나옴 (클래스 이름도 자연어)
- 레이영상 돌려서: license_plate 검출 + tail_light 검출 + 후미등을 미러/주유구로 오인 안 하는지 확인
- 4종이 안 무너졌는지도 확인

### 튜닝
- 후미등 약하거나 미러/주유구 오인 계속되면 → RAY_REPEAT 올리기(8→12)
- 기존 클래스 흔들리면 → RAY_REPEAT 낮추거나 REHEARSAL_PER_CLASS 올리기
- 새 학습이라 v3 대비 초반 성능 낮을 수 있음 (에포크 더 필요)